# Electron tomography workflow (rg-tomo, Python)

Python counterpart of `tomography.m` (rg-tomo-matlab). Run the cells from top to bottom; like the MATLAB
script, some sections are alternatives or optional, so skip the ones you do not need.

**Conventions (differences with MATLAB)**

* A tilt series is a `TiltSeries` with `data` of shape `(n_angles, n_rows, n_cols)`: projection `k` is
  `series.data[k]` (MATLAB `series.data(:,:,k+1)`). Angles are in degrees, the pixel size in nm.
* All indices are **0-based**, as shown by the stackview sliders (MATLAB projection 27 is index 26 here).
  Row ranges are `(start, stop)` with `stop` excluded.
* Views use [stackview](https://github.com/haesleinhuepf/stackview) (`slice_viewer` replaces `sliceViewer`,
  `orthoslice_viewer` replaces `orthosliceViewer`).
* The interactive tools (crop, tilt axis, manual alignment, cupping correction, image review) cannot block the
  notebook like MATLAB's `uiwait`: the cell shows the GUI, and the **next cell** collects the output with
  `gui.result()`.
* Reconstructions are done with ASTRA (EM/SIRT/FBP on the GPU), reconstructions are `(n_rows, n_cols, n_cols)`:
  slices along the tilt axis, written to `.rec` with the same layout as the MATLAB toolbox.
* Not ported (on purpose): CNN denoising, the 360° rendering movie and the surface extraction/export.

---
# LIBRARIES

In [1]:
import sys
TOOLBOX = r"C:\Users\rgirod\Data\rg-tomo2026-python"   # folder containing the rgtomo package
if TOOLBOX not in sys.path:
    sys.path.insert(0, TOOLBOX)

import numpy as np
import matplotlib.pyplot as plt
import rgtomo as rgt
%matplotlib inline

print("rgtomo", rgt.__version__, "| ASTRA CUDA available:", rgt.has_cuda())

rgtomo 0.1.0 | ASTRA CUDA available: True


---
## DATA PREPROCESSING
### Loading data
Loading has three options:
- `'fast'` assumes one file `TomoX.emi` and a corresponding `TomoX.log`, otherwise provide an angle file
  (a multi-frame Velox `.emd` is also accepted)
- `'manual'` assumes one folder and one `.emi`/`.ser` or `.emd` per angle
- `'tif'` assumes one `series.tif` and one `angle.csv` (or a folder of `.tif` files with the angles in their names)

In [2]:
fname = r"C:\Users\rgirod\Data\rg-tomo2026-python\data\LCysSeeds_P2_tiltSeries.tiff"
fangles = r"C:\Users\rgirod\Data\rg-tomo2026-python\data\LCysSeeds_P2_tiltAngles.csv"     # angles file, optional, None if unused
ftype = 'tif'      # 'fast', 'manual', 'tif'

series_raw = rgt.uniloader(fname, fangles, ftype)

series_raw.pixel_size = 0.5
print(series_raw)
rgt.slice_viewer(series_raw)

Loaded 52 projections with 1.000 nm pixel size
TiltSeries(52 projections of 227 x 169 px, pixel size 0.5 nm, angles -77.0..75.9 deg, dtype float32)


### Preprocessing and initial alignment
**Contrast.** The toolbox assumes HAADF-STEM contrast (bright object on a dark background): the automatic crop,
the background removal and the self-consistency alignment (positivity, centre of mass) rely on it. For
bright-field or ptychography data (dark object on a bright background, like this dataset) set
`invert_contrast = True`.

**Binning.** Think about your required resolution, a rule of thumb is to have at least 4-5 pixels for the
smallest features to be analyzed. It is not advised to keep more than this for size and noise considerations.

**Alignment.** For isolated particles, it works best to select one angle to which all projections will be
aligned. For slab-like cases, alignment to the previous image typically works better.

**Denoising.** A simple gaussian is often sufficient here.

In [ ]:
# Parameters
invert_contrast = False     # (bool): True for bright-field data, False for HAADF-STEM
rescale_factor = 1          # (float): Factor of the new pixel size
slice_to_align_to = None    # (int): Projection index (0-based) or None to align to neighbor
window = False              # (bool): adds a hamming window, can help if the ROI is not well isolated
edge = 2                    # (int): gaussian sigma after sobel filtering. if 0: no edge filter
iterations = 2              # (int): Iterations for x-correlation, typically 1-3
slab_like = False           # (bool):  if slab-like will stretch following 1/cos
y_range = None              # (start, end) or None: rows used to find the shifts: 
sigma_gauss_filter = 1      # (float): Denoising with gaussian filter, 0 for none

# Run code
series = series_raw.copy()
if invert_contrast:
    series.data = 1.0 - rgt.rescale(series.data)

# normalize each projection -> not always a correct assumption!
series.data = rgt.normalize_intensities(series.data)

if rescale_factor < 1:
    series = rgt.imresize_stack(series, rescale_factor)
    print(f"new pixel size: {series.pixel_size:.3f} nm")

series.data = rgt.gaussfilter_slice(series.data, sigma_gauss_filter)

stack = None
if iterations > 0:
    series.data, _, _, stack = rgt.align_stack_xcorr(series, slice_to_align_to, window, edge,
                                                     iterations, slab_like, 1, y_range)
series.data = rgt.rescale(series.data)

In [ ]:
# Check alignment: aligned series (top) and the filtered images used to find the shifts (bottom)
rgt.slice_viewer(series.data if stack is None else np.concatenate([series.data, rgt.rescale(stack)], axis=1))

## Crop and rotate
The tilt series must be rotated to be vertical. Here we only correct for the fact that the scan direction has
typically a 90° difference between the Osiris and the Titans.
**/!\ rotate by 1x90° only for the Titans, 0 for the Osiris.**

Draw the ROI on the projection (or use the sliders), then run the next cell to crop.

In [ ]:
rotate = 0   # rotation of n times 90°

series_test = series.copy()
series_test.data = rgt.rot90_stack(series_test.data, rotate)
crop_gui = rgt.crop_stack_gui(series_test)
crop_gui

In [ ]:
series_test = crop_gui.result()
print("ROI (x0, y0, width, height):", crop_gui.rect, "->", series_test)

---
# FAST TOMO PROCESSING
## Remove bad images - Ajinkya's (recommended)
This script will compute a reconstruction and reproject it, then compare the original and simulated tilt-series
with SSIM. Only the original projections that best match with the reprojections will be kept.

In [ ]:
series_retained = rgt.generate_clean_tilt_series(series_test, plot=True)
rgt.slice_viewer(series_retained)

## Remove bad images - Wouter's (not recommended)
This script measures the error between consecutive frames, under the assumption that a distorted image will
have a high error with its neighbors.

In [ ]:
RETAIN_AT_LEAST = 100
ACCEPTANCE_MULTIPLIER = 2.0

series_retained = rgt.generate_clean_series(series_test, RETAIN_AT_LEAST, ACCEPTANCE_MULTIPLIER)
rgt.slice_viewer(series_retained)

## Remove bad images - Manual (when everything else fails)
Indices are 0-based, as shown by the viewer slider. Run this cell only once (indices refer to `series_test`
before the removal).

In [ ]:
exclude_list = [26]   # MATLAB [27]

series_test = rgt.select_images(series_test, exclude_list)
rgt.slice_viewer(series_test)

Alternatively, review the projections one by one (Keep / Remove):

In [ ]:
select_gui = rgt.select_images_gui(series_test)
select_gui

In [ ]:
series_test = select_gui.result()
print("removed:", select_gui.excluded, "->", series_test)

---
# FINAL PROCESSING
## Final background removal and second alignment
**Background removal** (optional): the script will fit a background function, subtract it, and remove parts
that are below an intensity threshold. Decrease `BGSUBTRACT_THRESHOLD` to remove less.

**Slab-like** parameters when working with continuous geometries (not isolated particles): only use if you
already know the tilt axis to be more or less vertical. Will apply a stretching function only for alignment
that compensates the fact that new areas are entering the ROI at high angles.
**/!\ this assumes the tilt axis is vertical, it may be needed to iterate a few times between this alignment
and the tilt-axis one.**

As in `tomography.m`, the cleaned series is used for `'fast'` data, otherwise `series_test`.

In [ ]:
remove_background = True        # (bool)
BGSUBTRACT_THRESHOLD = 0.7      # factor of otsu graythresh to keep, higher removes more, default 0.5

slice_to_align_to = None        # Projection index (0-based) or None to align to neighbor
window = False                  # (bool) Adds a hamming window, can help if the ROI is not well isolated
iterations = 0                  # Iterations for x-correlation, typically 1-3
slab_like = False               # (bool), will apply a 1/cos(theta) stretch

# Run code
series_final = (series_retained if ftype == 'fast' else series_test).copy()

if remove_background:
    series_final.data = rgt.background_subtract(series_final.data, BGSUBTRACT_THRESHOLD)
    series_final.data = rgt.normalize_intensities(series_final.data)   # norm each proj to mean 1 std 1
series_final.data = rgt.rescale(series_final.data)
if iterations > 0:
    series_final.data, _, _, stack = rgt.align_stack_xcorr(series_final, slice_to_align_to, window,
                                                           iterations=iterations, slab_like=slab_like)

# Show results
# plt.plot(series_final.angles, "k."); plt.title("Projection angles"); plt.show()
rgt.slice_viewer(rgt.rescale(series_final.data))

## Align tilt axis
Three slices are reconstructed: shift/rotate the tilt axis (red line) and press *Preview* until the arcs of a
misaligned axis disappear. Run the next cell to apply. (This tool can be used at any stage, e.g. again after the
self-consistency refinements.)

In [ ]:
tilt_gui = rgt.align_tilt_axis_gui(series_final)
tilt_gui

In [ ]:
series_final = tilt_gui.result()
print(f"tilt axis translation {tilt_gui.translation} px, rotation {tilt_gui.rotation} deg")

## Refinement with vertical mass fluctuation / common line
This script projects each image on the Y-axis. If the tilt axis is well aligned, these curves should be equal
and the Y shifts can be determined by minimizing this error.

In [ ]:
high_pass_filter = 0.01        # band pass filter, lower = more filtering, 0.005-0.01 needed for STEM

series_final_self = rgt.csaxs_vmf(series_final, high_pass_filter)
series_final_self.data = rgt.rescale(series_final_self.data)

rgt.slice_viewer(series_final_self)

## Refinement with features-based tracking
This script detects blobs and finds the best translation to register them (SIFT features here, SURF in MATLAB);
there are many internal parameters, see `help(rgt.align_stack_features)`.

In [ ]:
series_final_self.data = rgt.align_stack_features(series_final_self.data)

rgt.slice_viewer(series_final_self)

## Refinement with self-consistency - x
The projections are reconstructed and reprojected iteratively and shifted to match the reprojections (on 4x and
2x binned data). `vert_range` restricts the rows used, e.g. `(100, 300)`.

In [ ]:
vert_range = None
series_final_self2 = rgt.align_tomo_consistency(series_final_self, vert_range, 0.005)

# left: before, right: after
rgt.slice_viewer(np.concatenate([series_final_self.data, series_final_self2.data], axis=2))

## Refinement with self-consistency - y

In [ ]:
series_final_self3 = rgt.align_tomo_consistency(series_final_self2, vert_range, 0.01,
                                                align_horizontal=False, align_vertical=True)
rgt.slice_viewer(np.concatenate([series_final_self2.data, series_final_self3.data], axis=2))

## Refinement with self-consistency - geom
This needs a relaxed min step size, e.g., 0.2 when pure x align can go down to 0.05. Like in `tomography.m`
this starts from `series_final_self2` (use `series_final_self3` to keep the vertical refinement). The refined
in-plane rotation of each projection is reported (`meta['consistency_tilt']`); as in MATLAB only the shifts
are applied, pass `apply_geometry=True` to also rotate the projections.

In [ ]:
series_final_self4 = rgt.align_tomo_consistency(series_final_self2, vert_range, 0.005,
                                                align_horizontal=True, align_vertical=False,
                                                refine_geometry=True)
rgt.slice_viewer(series_final_self4)

## Finalize the alignment manually if required
Green: image *i*, magenta: image *i*+1. Shift with the arrow buttons, or the keyboard with the mouse over the
image (arrows, A/D for previous/next, PageUp/PageDown to change the step). Run the next cell to apply.

In [ ]:
manual_gui = rgt.align_stack_manual_gui(series_final_self4)
manual_gui

In [ ]:
series_final_self4 = manual_gui.result()

## Cupping correction (optional, for large particles)
Increase the correction strength until the reconstructed slice (EM/SIRT) has no darker core.

In [ ]:
cupping_gui = rgt.cupping_correction_gui(series_final_self4)
cupping_gui

In [ ]:
series_final_self4 = cupping_gui.result()
# series_final_self4.data = rgt.remove_bkg_median(series_final_self4.data)

---
# RECONSTRUCTION

In [ ]:
alg = 'sirt'   # 'em' or 'sirt' ('fbp' also available)
it = 50        # iterations, 25 for EM, up to 50-100 for SIRT

series_rec = series_final_self4                          # tilt series to reconstruct
result = rgt.reconstruct(series_rec, alg, it, True, 30)  # apodization, 30 px padding
rgt.orthoslice_viewer(result)

---
# POST-PROCESSING
## Align and post process
Simple filtering in 3D + alignment of the major axis of reconstruction using the first principal component.
If the reconstruction is not a rod, leave the PCA line commented.

In [ ]:
gaussian_sigma = 2            # std of the gaussian kernel, in vx
median_kernel = (1, 1, 1)     # size of the median kernel

result_processed = result.astype(np.float32)
result_processed = rgt.gaussian_filter_3d(result_processed, gaussian_sigma)
# result_processed = rgt.median_filter_3d(result_processed, median_kernel)
# result_processed = rgt.align_z_pca(result_processed, (1, 99))  # lower and upper percentile thresholds for outliers

rgt.orthoslice_viewer(result_processed)

## Compute rotational slices if desired
This script will take a central orthoslice through the reconstruction at different rotation angles about the
major axis (the first axis of the volume). It is particularly convenient to visualize the wrinkles in chiral NRs.

In [ ]:
spacing = 2   # Spacing in deg

slices = rgt.rotational_slices(result_processed, spacing)
rgt.slice_viewer(slices)

---
# EXPORTS
## Save reconstruction
An automated naming is proposed, which assumes that the input file or folder was named
`\\yourTitan\sample\TomoX.ser` and exports files to `\\save_folder\sample\sample_PX_alg.rec` where P stands for
"particle", X is the acquisition number, alg is the algorithm used for reconstruction.

Otherwise, you can provide a root and all files will be exported to `\\save_folder\root_alg.rec`
(the default root below differs from the MATLAB one so the MATLAB results in the same folder are not overwritten).

In [ ]:
save_folder = r"C:\Users\rgirod\Data\20261006_Titan1_80kV_PtychoTomo2deg\2-Tomo1\Tiffs"
save_all = 2              # 0: only tomo rec, 1: tomo and tomo processed, 2: tomo, aligned tilt series, csv, slices if exist
from_root = 'Rec_inv_py'  # Option to override automated naming, None for the automated naming

# Naming and folder management
rec_name = rgt.get_series_names(fname, save_folder, from_root)
if 'alg' in globals():
    rec_name = f"{rec_name}{alg}{it}"
print("saving rec:", rec_name)

# Saving
if 'result' in globals():
    rgt.write_rec(result, rec_name + '.rec', series_rec.pixel_size)
if save_all > 0 and 'result_processed' in globals():
    rgt.write_rec(result_processed, rec_name + '_processed.rec', series_rec.pixel_size)
if save_all > 1:
    rgt.write_tiff(series_rec.data, rec_name + '_tiltSeries.tiff', series_rec.pixel_size)
    rgt.write_angles_csv(series_rec.angles, rec_name + '_tiltAngles.csv')
    if 'slices' in globals():
        rgt.write_rec(slices, rec_name + '_rotSlices.rec', series_rec.pixel_size)